# Advanced Problems with Solutions: `yield from`, `throw()`, and Generator Exception Delegation

This notebook is a problem-driven deep dive into Python generator delegation.

## Learning goals

You will learn to:

- trace `next()`, `send()`, `throw()`, and `close()` through `yield from`;
- distinguish yielded values from generator return values;
- retrieve `StopIteration.value`;
- design recoverable exception commands;
- understand `GeneratorExit` and cleanup rules;
- route exceptions through nested delegation;
- build transaction-like and supervisor-like coroutines;
- compare native `yield from` with manual forwarding;
- test generator protocols with precise assertions.

## Best practices used throughout

1. Prime generators explicitly with `next(...)`.
2. Use narrow, purpose-specific exception classes.
3. Put cleanup in `finally`.
4. Never yield after `GeneratorExit`.
5. Keep the protocol state explicit.
6. Return structured data instead of depending on printed output.
7. Test every transition.
8. Prefer native `yield from` in production.


In [10]:
from __future__ import annotations

from dataclasses import dataclass
from io import StringIO
from numbers import Real
from typing import Any, Callable, Generator
import csv


## Shared helpers

The helpers below capture a generator's return value from `StopIteration.value`.


In [11]:
def finish_with_throw(gen: Generator, exc: BaseException) -> Any:
    try:
        gen.throw(exc)
    except StopIteration as stop:
        return stop.value
    raise AssertionError("The generator did not terminate as expected.")


def finish_with_next(gen: Generator) -> Any:
    try:
        next(gen)
    except StopIteration as stop:
        return stop.value
    raise AssertionError("The generator did not terminate as expected.")


# Problem 1 — Trace the complete delegation protocol

Create a subgenerator that yields readiness, acknowledges sent values, handles a recoverable exception, and records cleanup. Wrap it in a delegator and prove that `send()`, `throw()`, and `close()` reach the active subgenerator.


## Solution 1


In [12]:
class Recover(Exception):
    pass


def traced_worker(trace: list[str]) -> Generator[str, object, None]:
    try:
        value = yield "READY"
        while True:
            try:
                trace.append(f"worker received: {value!r}")
                value = yield f"ACK:{value}"
            except Recover as exc:
                trace.append(f"worker recovered from: {exc}")
                value = yield f"RECOVERED:{exc}"
    finally:
        trace.append("worker cleanup")


def traced_delegator(trace: list[str]) -> Generator[str, object, None]:
    trace.append("delegator start")
    try:
        yield from traced_worker(trace)
    finally:
        trace.append("delegator cleanup")


In [13]:
trace: list[str] = []
gen = traced_delegator(trace)

assert next(gen) == "READY"
assert gen.send(10) == "ACK:10"
assert gen.throw(Recover("temporary issue")) == "RECOVERED:temporary issue"
assert gen.send(7) == "ACK:7"
gen.close()

assert trace == [
    "delegator start",
    "worker received: 10",
    "worker recovered from: temporary issue",
    "worker received: 7",
    "worker cleanup",
    "delegator cleanup",
]
trace


['delegator start',
 'worker received: 10',
 'worker recovered from: temporary issue',
 'worker received: 7',
 'worker cleanup',
 'delegator cleanup']

### Key idea

While the delegator is suspended at `yield from`, protocol operations are forwarded to the active subgenerator. The delegator regains control when the subgenerator returns or raises an exception it does not handle.


# Problem 2 — Capture a delegated return value

Design a worker that terminates when it receives `StopWorker`, returns a structured summary, and is wrapped by an outer delegator that adds metadata. Retrieve the final value at the caller.


## Solution 2


In [14]:
class StopWorker(Exception):
    pass


def count_worker() -> Generator[int, None, dict[str, object]]:
    count = 0
    while True:
        try:
            yield count
            count += 1
        except StopWorker as exc:
            return {"count": count, "reason": str(exc)}


def count_service() -> Generator[int, None, dict[str, object]]:
    worker_result = yield from count_worker()
    return {"service": "count-service", "worker_result": worker_result}


In [15]:
gen = count_service()
assert next(gen) == 0
assert next(gen) == 1
assert next(gen) == 2

result = finish_with_throw(gen, StopWorker("maintenance"))
assert result == {
    "service": "count-service",
    "worker_result": {"count": 2, "reason": "maintenance"},
}
result


{'service': 'count-service',
 'worker_result': {'count': 2, 'reason': 'maintenance'}}

A generator's `return value` becomes `StopIteration(value)`. A `yield from` expression catches that internal exception and evaluates to its `.value`.


# Problem 3 — Build a recoverable state machine

Implement a running-total coroutine. Sending a number updates the total; throwing `ResetTotal` resets it without terminating. Use one predictable suspension point.


## Solution 3


In [16]:
class ResetTotal(Exception):
    pass


def running_total() -> Generator[object, Real, None]:
    total: Real = 0
    output: object = "READY"

    while True:
        try:
            value = yield output
        except ResetTotal:
            total = 0
            output = {"event": "reset", "total": total}
        else:
            if isinstance(value, bool) or not isinstance(value, Real):
                raise TypeError("expected a real number, excluding bool")
            total += value
            output = {"event": "value", "total": total}


In [17]:
gen = running_total()
assert next(gen) == "READY"
assert gen.send(5) == {"event": "value", "total": 5}
assert gen.send(2.5) == {"event": "value", "total": 7.5}
assert gen.throw(ResetTotal()) == {"event": "reset", "total": 0}
assert gen.send(4) == {"event": "value", "total": 4}
gen.close()


Using an `output` state variable avoids adding a second, easy-to-misunderstand `yield` inside an exception handler.


# Problem 4 — Catch a subgenerator failure in the delegator

A primary worker raises `ValueError` for `"boom"`. The delegator must catch it, yield a transition message, switch to a fallback worker, and continue.


## Solution 4


In [18]:
def primary_worker() -> Generator[str, str, None]:
    value = yield "PRIMARY_READY"
    while True:
        if value == "boom":
            raise ValueError("primary worker failed")
        value = yield f"primary:{value}"


def fallback_worker() -> Generator[str, str, None]:
    value = yield "FALLBACK_READY"
    while True:
        value = yield f"fallback:{value}"


def resilient_service() -> Generator[str, str, None]:
    try:
        yield from primary_worker()
    except ValueError as exc:
        yield f"SWITCHED:{exc}"
        yield from fallback_worker()


In [19]:
gen = resilient_service()
assert next(gen) == "PRIMARY_READY"
assert gen.send("alpha") == "primary:alpha"
assert gen.send("boom") == "SWITCHED:primary worker failed"
assert next(gen) == "FALLBACK_READY"
assert gen.send("beta") == "fallback:beta"
gen.close()


A delegator is a useful exception boundary for fallback, logging, translation, or restart policy.


# Problem 5 — Correct and incorrect `GeneratorExit` handling

Part A proves that `close()` triggers nested cleanup. Part B proves that yielding after `GeneratorExit` raises `RuntimeError`.


## Solution 5A — Correct cleanup


In [20]:
def safe_worker(log: list[str]) -> Generator[str, None, None]:
    try:
        while True:
            yield "tick"
    finally:
        log.append("safe worker cleanup")


def safe_outer(log: list[str]) -> Generator[str, None, None]:
    try:
        yield from safe_worker(log)
    finally:
        log.append("safe outer cleanup")


In [21]:
log: list[str] = []
gen = safe_outer(log)
assert next(gen) == "tick"
gen.close()
assert log == ["safe worker cleanup", "safe outer cleanup"]
log


['safe worker cleanup', 'safe outer cleanup']

## Solution 5B — Illegal yield while closing


In [22]:
def bad_worker() -> Generator[int, None, None]:
    try:
        yield 1
    except GeneratorExit:
        yield 2


In [23]:
gen = bad_worker()
assert next(gen) == 1

error_message = None
try:
    gen.close()
except RuntimeError as exc:
    error_message = str(exc)
finally:
    try:
        gen.close()
    except RuntimeError:
        pass

assert error_message == "generator ignored GeneratorExit"
error_message


'generator ignored GeneratorExit'

Put cleanup in `finally`. Never suppress `GeneratorExit` and then yield another value.


# Problem 6 — Snapshot a running average into CSV

Build an averaging coroutine that receives real numbers, yields the current average, handles `Snapshot` injected with `throw()`, writes CSV snapshots, and continues running. Use `StringIO` so tests have no filesystem dependency.


## Solution 6


In [24]:
class Snapshot(Exception):
    def __init__(self, label: str):
        super().__init__(label)
        self.label = label


def csv_averager(stream: StringIO) -> Generator[float | None, Real, None]:
    writer = csv.writer(stream)
    writer.writerow(["label", "count", "average"])

    total = 0.0
    count = 0
    average: float | None = None

    while True:
        try:
            value = yield average
        except Snapshot as command:
            if count:
                writer.writerow([command.label, count, average])
                stream.flush()
        else:
            if isinstance(value, bool) or not isinstance(value, Real):
                raise TypeError("expected a real number, excluding bool")
            total += float(value)
            count += 1
            average = total / count


In [25]:
stream = StringIO()
gen = csv_averager(stream)

assert next(gen) is None
assert gen.send(10) == 10.0
assert gen.send(20) == 15.0
assert gen.throw(Snapshot("checkpoint-A")) == 15.0
assert gen.send(30) == 20.0
assert gen.throw(Snapshot("checkpoint-B")) == 20.0

rows = list(csv.reader(StringIO(stream.getvalue())))
assert rows == [
    ["label", "count", "average"],
    ["checkpoint-A", "2", "15.0"],
    ["checkpoint-B", "3", "20.0"],
]
gen.close()
rows


[['label', 'count', 'average'],
 ['checkpoint-A', '2', '15.0'],
 ['checkpoint-B', '3', '20.0']]

Exception commands are expressive, but ordinary typed messages are often clearer for non-exceptional operations.


# Problem 7 — Route exceptions through three delegation levels

Create `outer_service -> middle_service -> leaf_worker`.

- `Audit` is handled by the leaf.
- `Shutdown` propagates through the leaf.
- The middle layer catches `Shutdown` and returns a summary.
- The outer layer wraps that summary.


## Solution 7


In [26]:
class Audit(Exception):
    def __init__(self, tag: str):
        super().__init__(tag)
        self.tag = tag


class Shutdown(Exception):
    pass


def leaf_worker() -> Generator[str, str, None]:
    output = "LEAF_READY"
    while True:
        try:
            value = yield output
        except Audit as command:
            output = f"AUDIT:{command.tag}"
        else:
            output = f"leaf:{value}"


def middle_service() -> Generator[str, str, dict[str, str]]:
    try:
        yield from leaf_worker()
    except Shutdown as exc:
        return {"middle": "stopped", "reason": str(exc)}


def outer_service() -> Generator[str, str, dict[str, object]]:
    middle_result = yield from middle_service()
    return {"outer": "complete", "middle_result": middle_result}


In [27]:
gen = outer_service()
assert next(gen) == "LEAF_READY"
assert gen.send("A") == "leaf:A"
assert gen.throw(Audit("manual-check")) == "AUDIT:manual-check"
assert gen.send("B") == "leaf:B"

result = finish_with_throw(gen, Shutdown("planned shutdown"))
assert result == {
    "outer": "complete",
    "middle_result": {"middle": "stopped", "reason": "planned shutdown"},
}
result


{'outer': 'complete',
 'middle_result': {'middle': 'stopped', 'reason': 'planned shutdown'}}

`throw()` first enters the deepest active subgenerator. If unhandled, normal stack unwinding continues through each delegator.


# Problem 8 — Implement a transaction-style coroutine

Ordinary sent values are staged. Injected commands perform:

- `Commit`: apply and clear staged values;
- `Rollback`: discard staged values;
- `Abort`: terminate and return a summary.


## Solution 8


In [28]:
class Commit(Exception):
    pass


class Rollback(Exception):
    pass


class Abort(Exception):
    pass


def transaction_worker(
    apply_batch: Callable[[list[object]], None],
) -> Generator[dict[str, object], object, dict[str, object]]:
    pending: list[object] = []
    committed_count = 0
    output: dict[str, object] = {
        "event": "ready",
        "pending": 0,
        "committed": 0,
    }

    while True:
        try:
            item = yield output
        except Commit:
            batch = pending.copy()
            apply_batch(batch)
            committed_count += len(batch)
            pending.clear()
            output = {
                "event": "commit",
                "size": len(batch),
                "pending": 0,
                "committed": committed_count,
            }
        except Rollback:
            discarded = len(pending)
            pending.clear()
            output = {
                "event": "rollback",
                "discarded": discarded,
                "pending": 0,
                "committed": committed_count,
            }
        except Abort as exc:
            return {
                "event": "aborted",
                "reason": str(exc),
                "pending_discarded": len(pending),
                "committed": committed_count,
            }
        else:
            pending.append(item)
            output = {
                "event": "staged",
                "item": item,
                "pending": len(pending),
                "committed": committed_count,
            }


In [29]:
applied: list[object] = []

def apply_batch(batch: list[object]) -> None:
    applied.extend(batch)


gen = transaction_worker(apply_batch)
assert next(gen)["event"] == "ready"
assert gen.send("A")["pending"] == 1
assert gen.send("B")["pending"] == 2

assert gen.throw(Commit()) == {
    "event": "commit",
    "size": 2,
    "pending": 0,
    "committed": 2,
}
assert applied == ["A", "B"]

assert gen.send("C")["pending"] == 1
assert gen.throw(Rollback())["discarded"] == 1

final_result = finish_with_throw(gen, Abort("client cancelled"))
assert final_result == {
    "event": "aborted",
    "reason": "client cancelled",
    "pending_discarded": 0,
    "committed": 2,
}
final_result


{'event': 'aborted',
 'reason': 'client cancelled',
 'pending_discarded': 0,
 'committed': 2}

This is a protocol model, not full ACID behavior. Real transactions also need durability, retry safety, concurrency control, and crash recovery.


# Problem 9 — Reject the most recently staged record

Add a `RejectLast(reason)` command. It removes only the most recent pending item, records an audit entry, and keeps the coroutine alive.


## Solution 9


In [30]:
class RejectLast(Exception):
    def __init__(self, reason: str):
        super().__init__(reason)
        self.reason = reason


def audited_staging_area() -> Generator[
    dict[str, object], object, dict[str, object]
]:
    pending: list[object] = []
    audit: list[dict[str, object]] = []
    output: dict[str, object] = {"event": "ready", "pending": []}

    while True:
        try:
            item = yield output
        except RejectLast as command:
            rejected = pending.pop() if pending else None
            entry = {
                "event": "rejected",
                "item": rejected,
                "reason": command.reason,
            }
            audit.append(entry)
            output = {**entry, "pending": pending.copy()}
        except Abort as exc:
            return {
                "reason": str(exc),
                "pending": pending.copy(),
                "audit": audit.copy(),
            }
        else:
            pending.append(item)
            output = {
                "event": "staged",
                "item": item,
                "pending": pending.copy(),
            }


In [31]:
gen = audited_staging_area()
assert next(gen)["event"] == "ready"
assert gen.send({"id": 1})["pending"] == [{"id": 1}]
assert gen.send({"id": 2})["pending"] == [{"id": 1}, {"id": 2}]

rejection = gen.throw(RejectLast("failed validation"))
assert rejection == {
    "event": "rejected",
    "item": {"id": 2},
    "reason": "failed validation",
    "pending": [{"id": 1}],
}

result = finish_with_throw(gen, Abort("done"))
assert result["pending"] == [{"id": 1}]
assert result["audit"][0]["reason"] == "failed validation"
result


{'reason': 'done',
 'pending': [{'id': 1}],
 'audit': [{'event': 'rejected',
   'item': {'id': 2},
   'reason': 'failed validation'}]}

# Problem 10 — Manually reproduce the essential forwarding loop

For learning only, implement a simplified proxy that forwards `next()`, `send()`, `throw()`, `close()`, and the subgenerator return value. Compare it with native `yield from`.


## Solution 10


In [32]:
def manual_delegate(subgen: Generator) -> Generator[object, object, object]:
    try:
        current = next(subgen)
    except StopIteration as stop:
        return stop.value

    while True:
        try:
            sent = yield current
        except GeneratorExit:
            close = getattr(subgen, "close", None)
            if close is not None:
                close()
            raise
        except BaseException as exc:
            throw = getattr(subgen, "throw", None)
            if throw is None:
                raise
            try:
                current = throw(type(exc), exc, exc.__traceback__)
            except StopIteration as stop:
                return stop.value
        else:
            try:
                current = next(subgen) if sent is None else subgen.send(sent)
            except StopIteration as stop:
                return stop.value


In [33]:
class Ping(Exception):
    pass


class Finish(Exception):
    pass


def protocol_worker() -> Generator[object, int, dict[str, object]]:
    total = 0
    output: object = "READY"

    while True:
        try:
            value = yield output
        except Ping as exc:
            output = f"PONG:{exc}"
        except Finish as exc:
            return {"total": total, "reason": str(exc)}
        else:
            total += value
            output = total


def native_delegate() -> Generator[object, int, dict[str, object]]:
    result = yield from protocol_worker()
    return result


def educational_manual_delegate() -> Generator[object, int, dict[str, object]]:
    result = yield from manual_delegate(protocol_worker())
    return result


def drive_protocol(factory: Callable[[], Generator]) -> tuple[list[object], object]:
    gen = factory()
    outputs = [
        next(gen),
        gen.send(3),
        gen.throw(Ping("health-check")),
        gen.send(4),
    ]
    final = finish_with_throw(gen, Finish("complete"))
    return outputs, final


In [34]:
native_result = drive_protocol(native_delegate)
manual_result = drive_protocol(educational_manual_delegate)

assert native_result == manual_result
assert native_result == (
    ["READY", 3, "PONG:health-check", 7],
    {"total": 7, "reason": "complete"},
)
native_result


C:\Users\user1\AppData\Local\Temp\ipykernel_12948\4046155745.py:20: DeprecationWarning: the (type, exc, tb) signature of throw() is deprecated, use the single-arg signature instead.
  current = throw(type(exc), exc, exc.__traceback__)


(['READY', 3, 'PONG:health-check', 7], {'total': 7, 'reason': 'complete'})

`yield from` is a bidirectional protocol bridge, not merely a loop over produced values. Use native `yield from` in production.


# Problem 11 — Build a restartable supervisor

The worker accepts integers and tracks a local total. `QueryState` is handled by the worker. `RestartWorker` propagates to the supervisor, which starts a fresh worker. `StopSupervisor` returns a final summary.


## Solution 11


In [35]:
class QueryState(Exception):
    pass


class RestartWorker(Exception):
    pass


class StopSupervisor(Exception):
    pass


def supervised_worker(worker_id: int) -> Generator[object, int, None]:
    total = 0
    output: object = {
        "event": "ready",
        "worker_id": worker_id,
        "total": total,
    }

    while True:
        try:
            value = yield output
        except QueryState:
            output = {
                "event": "state",
                "worker_id": worker_id,
                "total": total,
            }
        else:
            if isinstance(value, bool) or not isinstance(value, int):
                raise TypeError("expected an integer, excluding bool")
            total += value
            output = {
                "event": "update",
                "worker_id": worker_id,
                "total": total,
            }


def supervisor() -> Generator[object, int, dict[str, object]]:
    restart_count = 0

    while True:
        try:
            yield from supervised_worker(restart_count)
        except RestartWorker:
            restart_count += 1
            continue
        except StopSupervisor as exc:
            return {
                "event": "stopped",
                "reason": str(exc),
                "restart_count": restart_count,
            }


In [36]:
gen = supervisor()
assert next(gen) == {
    "event": "ready",
    "worker_id": 0,
    "total": 0,
}
assert gen.send(5)["total"] == 5
assert gen.throw(QueryState())["total"] == 5

assert gen.throw(RestartWorker()) == {
    "event": "ready",
    "worker_id": 1,
    "total": 0,
}
assert gen.send(2)["total"] == 2

result = finish_with_throw(gen, StopSupervisor("deployment"))
assert result == {
    "event": "stopped",
    "reason": "deployment",
    "restart_count": 1,
}
result


{'event': 'stopped', 'reason': 'deployment', 'restart_count': 1}

The worker owns state transitions; the supervisor owns restart policy. Exception propagation keeps these concerns separate.


# Problem 12 — Capstone: command-driven metrics service

Build a nested service that:

- accepts real numbers;
- tracks count, total, mean, minimum, and maximum;
- handles labeled snapshots;
- handles resets without terminating;
- returns a final summary on stop;
- wraps the worker result in an outer service result.


## Solution 12


In [37]:
class MetricsSnapshot(Exception):
    def __init__(self, label: str):
        super().__init__(label)
        self.label = label


class ResetMetrics(Exception):
    pass


class StopMetrics(Exception):
    pass


def metrics_worker(
    snapshot_sink: list[dict[str, object]],
) -> Generator[dict[str, object], Real, dict[str, object]]:
    count = 0
    total = 0.0
    minimum: float | None = None
    maximum: float | None = None

    def current_metrics() -> dict[str, object]:
        return {
            "count": count,
            "total": total,
            "mean": None if count == 0 else total / count,
            "minimum": minimum,
            "maximum": maximum,
        }

    output: dict[str, object] = {
        "event": "ready",
        "metrics": current_metrics(),
    }

    while True:
        try:
            value = yield output
        except MetricsSnapshot as command:
            snapshot = {"label": command.label, **current_metrics()}
            snapshot_sink.append(snapshot)
            output = {"event": "snapshot", "snapshot": snapshot.copy()}
        except ResetMetrics:
            count = 0
            total = 0.0
            minimum = None
            maximum = None
            output = {"event": "reset", "metrics": current_metrics()}
        except StopMetrics as exc:
            return {
                "reason": str(exc),
                "metrics": current_metrics(),
                "snapshot_count": len(snapshot_sink),
            }
        else:
            if isinstance(value, bool) or not isinstance(value, Real):
                raise TypeError("expected a real number, excluding bool")

            number = float(value)
            count += 1
            total += number
            minimum = number if minimum is None else min(minimum, number)
            maximum = number if maximum is None else max(maximum, number)
            output = {"event": "update", "metrics": current_metrics()}


def metrics_service(
    snapshot_sink: list[dict[str, object]],
) -> Generator[dict[str, object], Real, dict[str, object]]:
    worker_result = yield from metrics_worker(snapshot_sink)
    return {
        "service": "metrics",
        "status": "closed",
        "worker_result": worker_result,
    }


In [38]:
snapshots: list[dict[str, object]] = []
gen = metrics_service(snapshots)

assert next(gen)["event"] == "ready"
assert gen.send(10)["metrics"]["mean"] == 10.0
assert gen.send(20)["metrics"]["mean"] == 15.0

snapshot_response = gen.throw(MetricsSnapshot("before-reset"))
assert snapshot_response["snapshot"] == {
    "label": "before-reset",
    "count": 2,
    "total": 30.0,
    "mean": 15.0,
    "minimum": 10.0,
    "maximum": 20.0,
}

assert gen.throw(ResetMetrics()) == {
    "event": "reset",
    "metrics": {
        "count": 0,
        "total": 0.0,
        "mean": None,
        "minimum": None,
        "maximum": None,
    },
}

assert gen.send(-5)["metrics"]["mean"] == -5.0
assert gen.send(5)["metrics"]["mean"] == 0.0

final_result = finish_with_throw(gen, StopMetrics("normal completion"))
assert final_result == {
    "service": "metrics",
    "status": "closed",
    "worker_result": {
        "reason": "normal completion",
        "metrics": {
            "count": 2,
            "total": 0.0,
            "mean": 0.0,
            "minimum": -5.0,
            "maximum": 5.0,
        },
        "snapshot_count": 1,
    },
}
final_result


{'service': 'metrics',
 'status': 'closed',
 'worker_result': {'reason': 'normal completion',
  'metrics': {'count': 2,
   'total': 0.0,
   'mean': 0.0,
   'minimum': -5.0,
   'maximum': 5.0},
  'snapshot_count': 1}}

# Additional challenge exercises

## Challenge A — Timeout policy

Add a `Timeout` command to the supervisor. Restart for the first two consecutive timeouts, then terminate with a summary on the third.

## Challenge B — Exception translation

Catch a leaf `ValueError` in the delegator and raise `InvalidMessageError(...) from exc`. Verify `__cause__`.

## Challenge C — Multiple owned generators

A single `yield from` delegates to one active child. Create a service that owns two children and closes both in `finally`.

## Challenge D — Randomized protocol testing

Generate random numeric values, snapshots, and resets. Compare the metrics coroutine against a simple reference model after every operation.

## Challenge E — Replace exception commands with typed messages

Rebuild the capstone with frozen dataclasses such as `Add`, `SnapshotCommand`, and `ResetCommand`. Compare readability and type-checking.


# Common pitfalls checklist

- Sending a non-`None` value before priming.
- Expecting `close()` to return the generator's return value.
- Confusing a yielded value with `StopIteration.value`.
- Catching `BaseException` too broadly and swallowing `GeneratorExit`.
- Yielding after `GeneratorExit`.
- Forgetting that `throw()` first reaches the deepest active subgenerator.
- Adding accidental extra suspension points inside exception handlers.
- Using exception commands when ordinary typed messages would be clearer.
- Testing only printed output instead of protocol transitions.
- Reimplementing `yield from` in production.


# Summary

`yield from` forwards a complete generator protocol:

- yielded values flow outward;
- `send()` values flow inward;
- `throw()` exceptions flow inward;
- unhandled exceptions propagate outward;
- `close()` propagates `GeneratorExit`;
- subgenerator return values become the value of the `yield from` expression.

Robust designs keep state explicit, cleanup unconditional, exception types narrow, and every transition tested.
